In [ ]:
import requests
import json
from notebookutils import mssparkutils
from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, TimestampType
import time
from datetime import datetime, timedelta, timezone

# Capture Start Time (for watermark)
execution_start_time = datetime.now(timezone.utc)
print(f"Execution Start Time: {execution_start_time}")

# --- Configuration ---
# Kusto Details
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database = "MonitoringEventhouse"

# Auth Details (Update with your SPN or Key Vault logic)
tenant_id = "9e929790-272d-4977-a2ab-301443c11ece"
client_id = "b5c04c9c-0588-418f-8f60-2d83d38cb635"
client_secret = "your-secret"

# Recommended: Use Key Vault
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

# API Base URL
fabric_api_base = "https://api.fabric.microsoft.com/v1"


StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 3, Finished, Available, Finished, False)

Execution Start Time: 2026-04-09 06:43:16.583049+00:00


In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve SPN token for scope {scope}: {e}")
        return None


def get_tokens():
    global fabric_token, kusto_token
    fabric_token = get_spn_token("https://api.fabric.microsoft.com/.default")
    kusto_token  = get_spn_token(f"{kusto_cluster_uri}/.default")
    if fabric_token and kusto_token:
        print("Successfully retrieved tokens.")
    else:
        raise Exception("Failed to retrieve essential tokens (Fabric/Kusto).")


StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 4, Finished, Available, Finished, False)

Successfully retrieved tokens.


In [3]:
# --- Item Type Mappings ---

# Maps item type → REST API endpoint segment
ITEM_ENDPOINT_MAP = {
    "ApacheAirflowJob":              "ApacheAirflowJobs",
    "CopyJob":                       "copyJobs",
    "CosmosDBDatabase":              "cosmosDbDatabases",
    "Dashboard":                     "dashboards",
    "DataAgent":                     "dataAgents",
    "DataFlow":                      "dataflows",
    "DataPipeline":                  "dataPipelines",
    "Environment":                   "environments",
    "Eventhouse":                    "eventhouses",
    "EventSchemaSet":                "eventSchemaSets",
    "GraphQLApi":                    "GraphQLApis",
    "KQLDashboard":                  "kqlDashboards",
    "KQLQueryset":                   "kqlQuerysets",
    "Lakehouse":                     "lakehouses",
    "Map":                           "maps",
    "MirroredAzureDatabricksCatalog": "mirroredAzureDatabricksCatalogs",
    "MirroredDatabase":              "mirroredDatabases",
    "MountedDataFactory":            "mountedDataFactories",
    "Notebook":                      "notebooks",
    "PaginatedReport":               "reports",
    "Reflex":                        "reflexes",
    "Report":                        "reports",
    "SemanticModel":                 "semanticModels",
    "SparkJobDefinition":            "sparkJobDefinitions",
    "SQLDatabase":                   "sqlDatabases",
    "UserDataFunction":              "UserDataFunctions",
    "VariableLibrary":               "VariableLibraries",
    "Warehouse":                     "warehouses",
}


# --- Define Helper Functions ---

def delete_fabric_item(workspace_id, item_id, item_kind, max_retries=2):
    """
    Deletes a specific item from a Fabric Workspace.
    API: DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/{endpoint}/{itemId}
    Returns: (status_message, error_code, description)
    """
    if not fabric_token:
        print("Skipping Delete: No Fabric token available.")
        return "DeleteFailed", "NoToken", "No Fabric token available"

    # Use specific endpoint if mapped, otherwise default to generic 'items'
    endpoint = ITEM_ENDPOINT_MAP.get(item_kind, "items")
    url = f"{fabric_api_base}/workspaces/{workspace_id}/{endpoint}/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type": "application/json"
    }
    
    last_exception = None
    
    for attempt in range(max_retries):
        try:
            response = requests.delete(url, headers=headers)
            
            # 200 OK or 204 No Content are typical success codes for DELETE
            if response.status_code in [200, 204]:
                return "DeletedItem", str(response.status_code), "Success"
            elif response.status_code == 404:
                print(f"SKIPPED: Item {item_id} not found (already deleted?).")
                return "404: File Not Found", "404", "Item Not Found"
            elif response.status_code == 401:
                # Unauthorized - Need to add permissions
                print(f"UNAUTHORIZED (401): No permission for item {item_id} in ws {workspace_id}.")
                return "401: Unauthorized", "401", response.text
            elif response.status_code == 429:
                # Handle Rate Limiting
                print(f"RATE LIMIT (429) Headers: {response.headers}")
                retry_after = int(response.headers.get("Retry-After", 60))
                print(f"RATE LIMIT (429): Waiting {retry_after} seconds before retry {attempt + 1}/{max_retries}...")
                time.sleep(retry_after)
                continue
            else:
                print(f"FAILED: Could not delete item {item_id}. Status: {response.status_code}, Response: {response.text}")
                return "DeleteFailed", str(response.status_code), response.text
                
        except Exception as e:
            print(f"EXCEPTION: Error deleting item {item_id}: {e}")
            last_exception = e
    
    print(f"FAILED: Max retries ({max_retries}) exceeded for item {item_id}.")
    desc = str(last_exception) if last_exception else "Max retries exceeded"
    return "DeleteFailed", "MaxRetries", desc

StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 5, Finished, Available, Finished, False)

In [ ]:
def get_alerts_to_process(upper_bound, lower_bound=None):
    try:
        print("Reading from Kusto query...")

        upper_bound_str = upper_bound.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

        if lower_bound is not None:
            # Loop iteration: use explicit lower bound passed from the previous run
            lower_bound_str = lower_bound.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            kusto_query = f"""
            AlertLogs
            | where ingestion_time() >= datetime('{lower_bound_str}')
            | where ingestion_time() < datetime('{upper_bound_str}')
            | summarize arg_max(ingestion_time(), WorkspaceName, wstime, data_itemName, AlertStatus, data_itemKind) by WorkspaceId, data_itemId
            | where AlertStatus in ("EmailSent", "NoEmail")
            | project WorkspaceName, WorkspaceId, wstime, data_itemKind, data_itemName, data_itemId
            """
        else:
            # Initial run: derive lower bound from DeleteLastRunTime table in Kusto
            kusto_query = f"""
            let lastRunParam = toscalar(DeleteLastRunTime | summarize max(LastRunTime));
            let startTimeFilter = iff(isnull(lastRunParam), ago(100d), lastRunParam);
            AlertLogs
            | where ingestion_time() >= startTimeFilter
            | where ingestion_time() < datetime('{upper_bound_str}')
            | summarize arg_max(ingestion_time(), WorkspaceName, wstime, data_itemName, AlertStatus, data_itemKind) by WorkspaceId, data_itemId
            | where AlertStatus in ("EmailSent", "NoEmail")
            | project WorkspaceName, WorkspaceId, wstime, data_itemKind, data_itemName, data_itemId
            """

        df_alerts = spark.read \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoQuery", kusto_query) \
            .option("accessToken", kusto_token) \
            .load()

        alerts_list = df_alerts.select(
            "WorkspaceId", "WorkspaceName", "wstime",
            "data_itemKind", "data_itemName", "data_itemId"
        ).collect()

        print(f"Found {len(alerts_list)} alerts to process.")
        return alerts_list

    except Exception as e:
        print(f"Error reading from Kusto: {e}")
        return []


StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 6, Finished, Available, Finished, False)

Reading from Kusto query...
Found 3 alerts to process.


In [ ]:
def process_alerts(alerts_list):
    results_to_log    = []
    exceptions_to_log = []
    deferred_deletion_list = []

    if not alerts_list:
        print("No alerts to process.")
        return results_to_log, exceptions_to_log

    if not fabric_token:
        raise Exception("Aborting Deletion Process: No Fabric token available.")
    if not kusto_token:
        raise Exception("Aborting Deletion Process: No Kusto token available.")

    print(f"Starting deletion loop for {len(alerts_list)} items...")

    for row in alerts_list:
        item_id   = row['data_itemId']
        ws_id     = row['WorkspaceId']
        ws_name   = row['WorkspaceName']
        item_kind = row['data_itemKind']

        print(f"Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}] in ws: {ws_name} ({ws_id})")
        status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)

        # Defer items that return 400 UnknownError (possible dependency issue)
        if code == "400" and "UnknownError" in desc:
            print(f"DEFERRING: Item {item_id} returned 400 UnknownError (possible dependency). Saving for retry.")
            deferred_deletion_list.append(row)
            continue

        if status == "DeletedItem" or "404" in status:
            results_to_log.append({
                "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                "wstime": row['wstime'], "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'], "data_itemId": item_id,
                "AlertStatus": status,
            })
        else:
            exceptions_to_log.append({
                "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                "wstime": row['wstime'], "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'], "data_itemId": item_id,
                "AlertStatus": status, "ErrorCode": code, "Description": desc,
            })

    # Retry deferred items after the initial pass
    if deferred_deletion_list:
        print(f"\n--- Retrying {len(deferred_deletion_list)} deferred items after initial pass ---")
        for row in deferred_deletion_list:
            item_id   = row['data_itemId']
            ws_id     = row['WorkspaceId']
            ws_name   = row['WorkspaceName']
            item_kind = row['data_itemKind']

            print(f"RETRY Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}]")
            status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)

            if status == "DeletedItem" or "404" in status:
                results_to_log.append({
                    "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                    "wstime": row['wstime'], "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'], "data_itemId": item_id,
                    "AlertStatus": status,
                })
            else:
                exceptions_to_log.append({
                    "WorkspaceName": ws_name, "WorkspaceId": ws_id,
                    "wstime": row['wstime'], "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'], "data_itemId": item_id,
                    "AlertStatus": status, "ErrorCode": code, "Description": desc,
                })

    print(f"Deletion processing complete. Success/404: {len(results_to_log)}, Exceptions: {len(exceptions_to_log)}")
    return results_to_log, exceptions_to_log


StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 7, Finished, Available, Finished, False)

Starting deletion loop for 3 items...
Deleting 'Notebook_1' (8640f667-7161-4444-a82f-715a482f6ad9) [Notebook] in ws: ab_demo_4 (b7c510d6-c7fb-4311-a627-8c49f3f28933)
SKIPPED: Item 8640f667-7161-4444-a82f-715a482f6ad9 not found (already deleted?).
Deleting 'Notebook_1' (cb8351de-f81a-4bd3-8f9e-31f765c90550) [Notebook] in ws: ab_demo_3 (7d8301d0-d8af-43b8-9c35-2397c92952a8)
Deleting 'lh' (509cdbf1-b4fc-4495-acfe-37640e574272) [Lakehouse] in ws: ab_demo_4 (b7c510d6-c7fb-4311-a627-8c49f3f28933)
Deletion processing complete. Success/404: 3, Exceptions: 0


In [ ]:
def log_results(results_to_log, exceptions_to_log):
    if results_to_log:
        print(f"Logging {len(results_to_log)} entries to AlertLogs...")
        log_df = spark.createDataFrame([Row(**r) for r in results_to_log])
        try:
            log_df.write \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("kustoCluster", kusto_cluster_uri) \
                .option("kustoDatabase", kusto_database) \
                .option("kustoTable", "AlertLogs") \
                .option("accessToken", kusto_token) \
                .option("tableCreateOptions", "CreateIfNotExist") \
                .mode("Append") \
                .save()
            print("Successfully logged alert statuses to AlertLogs.")
        except Exception as e:
            print(f"Failed to log to AlertLogs: {e}")
    else:
        print("No standard results to log.")

    if exceptions_to_log:
        print(f"Logging {len(exceptions_to_log)} exceptions to DeleteException...")
        exc_df = spark.createDataFrame([Row(**r) for r in exceptions_to_log])
        try:
            exc_df.write \
                .format("com.microsoft.kusto.spark.synapse.datasource") \
                .option("kustoCluster", kusto_cluster_uri) \
                .option("kustoDatabase", kusto_database) \
                .option("kustoTable", "DeleteException") \
                .option("accessToken", kusto_token) \
                .option("tableCreateOptions", "CreateIfNotExist") \
                .mode("Append") \
                .save()
            print("Successfully logged exceptions to DeleteException.")
        except Exception as e:
            print(f"Failed to log to DeleteException: {e}")
    else:
        print("No exceptions to log.")


def update_watermark():
    print(f"Updating DeleteLastRunTime watermark to {execution_start_time}...")
    try:
        time_schema = StructType([StructField("LastRunTime", TimestampType(), False)])
        time_df = spark.createDataFrame([(execution_start_time,)], schema=time_schema)
        time_df.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", "DeleteLastRunTime") \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        print("Successfully updated watermark.")
    except Exception as e:
        print(f"Failed to update LastRunTime: {e}")


StatementMeta(, 028c890e-4005-4f5b-8b02-b91eefad9dc8, 8, Finished, Available, Finished, False)

Logging 3 entries to AlertLogs...
Successfully logged alerts status to AlertLogs.
No exceptions to log.
Updating DeleteLastRunTime watermark to 2026-04-09 06:43:16.583049+00:00...
Successfully updated watermark.


In [ ]:
# --- Main Execution ---
get_tokens()

# Initial run
alerts_list = get_alerts_to_process(execution_start_time)
if alerts_list:
    results_to_log, exceptions_to_log = process_alerts(alerts_list)
    log_results(results_to_log, exceptions_to_log)
update_watermark()

# --- Continuous Polling: run every minute for 1 hour ---
polling_duration_hours   = 1
polling_interval_seconds = 60

loop_start_time = datetime.now(timezone.utc)
loop_end_time   = loop_start_time + timedelta(hours=polling_duration_hours)

print(f"\nStarting continuous polling until {loop_end_time.isoformat()}")

iteration = 0

while True:
    now = datetime.now(timezone.utc)
    if now >= loop_end_time:
        print(f"\nPolling duration of {polling_duration_hours}h reached. Stopping.")
        break

    iteration += 1

    # previous execution_start_time becomes the lower bound for this iteration
    prev_execution_start_time = execution_start_time
    execution_start_time = datetime.now(timezone.utc)

    print(f"\n[Iteration {iteration}] {execution_start_time.strftime('%H:%M:%S UTC')} | window: [{prev_execution_start_time.strftime('%H:%M:%S')} - {execution_start_time.strftime('%H:%M:%S')}]")

    alerts_list = get_alerts_to_process(execution_start_time, lower_bound=prev_execution_start_time)
    if alerts_list:
        results_to_log, exceptions_to_log = process_alerts(alerts_list)
        log_results(results_to_log, exceptions_to_log)

    if datetime.now(timezone.utc) < loop_end_time:
        print(f"Sleeping {polling_interval_seconds}s...")
        time.sleep(polling_interval_seconds)

# Update watermark once after the final iteration
update_watermark()
print("Continuous polling finished.")
